# 🗜️ LoRA Compress - Full Model Compression on Colab

Compress an entire model layer-by-layer using LoRA decomposition.
Results (compressed layers + metadata) are stored in Google Drive.

## What This Does
1. Loads SmolLM2-135M (or your model)
2. Compresses each linear layer to LoRA (BA decomposition)
3. Stores compressed layers in Drive
4. Can resume if Colab disconnects

**Time estimate:** 1-3 hours for full model (depending on quality target)

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/LoRA_Compress'
COMPRESSED_DIR = f'{DRIVE_BASE}/compressed_models/full_model'

os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(COMPRESSED_DIR, exist_ok=True)
os.makedirs(f'{COMPRESSED_DIR}/layers', exist_ok=True)

print(f"✅ Drive mounted")
print(f"📁 Compressed model will be stored at: {COMPRESSED_DIR}")

## Step 2: Clone Repository & Install Dependencies

In [ ]:
%cd /content
!git clone https://github.com/qades/loracompress.git
%cd loracompress
!pip install -q transformers torch optuna tqdm

import torch
print(f"\n🔥 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## Step 3: Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════

# Model to compress
MODEL_NAME = 'HuggingFaceTB/SmolLM2-135M'

# Quality targets
MAX_ERROR = 8.0          # Maximum L1 error % (lower = better quality)
MIN_COMPRESSION = 10.0   # Minimum compression ratio (higher = smaller)

# Training settings
DEFAULT_RANK = 16       # Default LoRA rank
DEFAULT_EPOCHS = 500    # Training epochs per layer
DEFAULT_LR = 0.01       # Learning rate

# Resume from previous run?
RESUME = True

# Limit layers (for testing, set to None for all)
LIMIT_LAYERS = None

print("⚙️ Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Max error: {MAX_ERROR}%")
print(f"  Min compression: {MIN_COMPRESSION}x")
print(f"  Resume: {RESUME}")
print(f"  Limit: {LIMIT_LAYERS or 'All layers'}")

## Step 4: Load Model & Identify Layers to Compress

In [ ]:
from transformers import AutoModelForCausalLM
import json

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float32
)
model.eval()

# Find compressible layers
layers_to_compress = []
for name, param in model.named_parameters():
    # Target linear projection layers
    if any(x in name for x in ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']):
        if len(param.shape) == 2:
            layers_to_compress.append({
                'name': name,
                'shape': list(param.shape),
                'param_count': param.numel()
            })

if LIMIT_LAYERS:
    layers_to_compress = layers_to_compress[:LIMIT_LAYERS]

total_params = sum(l['param_count'] for l in layers_to_compress)

print(f"\n📊 Found {len(layers_to_compress)} layers to compress")
print(f"   Total parameters: {total_params/1e6:.1f}M")
print(f"\nFirst 5 layers:")
for l in layers_to_compress[:5]:
    print(f"  {l['name']}: {l['shape']}")

## Step 5: Compression Functions

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import os
from pathlib import Path

def compute_l1_error(W_approx, target):
    """L1 relative error: mean(|diff|) / mean(|target|) * 100"""
    l1_error = torch.mean(torch.abs(W_approx - target)).item()
    mean_abs_target = torch.mean(torch.abs(target)).item()
    return (l1_error / mean_abs_target * 100) if mean_abs_target > 0 else float('inf')

def compress_layer(target_weight, rank, lr, epochs, device='cuda', patience=50):
    """Compress a single layer using LoRA."""
    d, k = target_weight.shape
    target = target_weight.float().to(device)
    
    A = nn.Parameter(torch.randn(rank, k, device=device) * 0.01)
    B = nn.Parameter(torch.randn(d, rank, device=device) * 0.01)
    optimizer = torch.optim.AdamW([A, B], lr=lr)
    
    best_loss = float('inf')
    best_A, best_B = None, None
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        W_approx = torch.matmul(B, A)
        loss = F.mse_loss(W_approx, target)
        
        if not torch.isfinite(loss):
            break
        
        loss.backward()
        optimizer.step()
        
        current = loss.item()
        if current < best_loss - 1e-9:
            best_loss = current
            best_A = A.detach().clone()
            best_B = B.detach().clone()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epoch >= 100 and epochs_no_improve >= patience:
            break
    
    with torch.no_grad():
        W_best = torch.matmul(best_B, best_A)
        l1_error = compute_l1_error(W_best, target)
    
    compression = (d * k) / (rank * (d + k))
    
    return {
        'A': best_A.cpu(),
        'B': best_B.cpu(),
        'error': l1_error,
        'compression': compression,
        'rank': rank,
        'original_shape': [d, k]
    }

def load_existing_compressed(compressed_dir):
    """Check which layers are already compressed."""
    metadata_file = f'{compressed_dir}/metadata.json'
    if not os.path.exists(metadata_file):
        return {}
    
    with open(metadata_file) as f:
        metadata = json.load(f)
    
    return metadata.get('layers', {})

print("✅ Compression functions defined")

## Step 6: Run Compression

In [ ]:
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}\n")

# Check for existing compressed layers
existing_layers = load_existing_compressed(COMPRESSED_DIR) if RESUME else {}
if existing_layers:
    print(f"📂 Found {len(existing_layers)} already compressed layers")

# Initialize metadata
metadata = {
    'model_name': MODEL_NAME,
    'compression_config': {
        'max_error': MAX_ERROR,
        'min_compression': MIN_COMPRESSION,
        'default_rank': DEFAULT_RANK,
        'default_epochs': DEFAULT_EPOCHS,
        'default_lr': DEFAULT_LR,
    },
    'layers': existing_layers.copy()
}

# Statistics
total_start = time.time()
compressed_count = len(existing_layers)
skipped_count = 0

print("\n" + "="*70)
print("🚀 Starting Full Model Compression")
print("="*70 + "\n")

for i, layer_info in enumerate(layers_to_compress):
    layer_name = layer_info['name']
    
    # Skip if already compressed and quality is good
    if layer_name in existing_layers:
        existing = existing_layers[layer_name]
        if existing.get('error', 100) <= MAX_ERROR:
            print(f"[{i+1}/{len(layers_to_compress)}] ⏭️  Skipping {layer_name} (already good)")
            skipped_count += 1
            continue
    
    print(f"\n[{i+1}/{len(layers_to_compress)}] 🗜️  Compressing {layer_name}...")
    
    # Get weight
    weight = None
    for name, param in model.named_parameters():
        if name == layer_name:
            weight = param.data
            break
    
    if weight is None:
        print(f"   ⚠️ Could not find weight")
        continue
    
    # Compress
    start = time.time()
    result = compress_layer(weight, DEFAULT_RANK, DEFAULT_LR, DEFAULT_EPOCHS, device)
    elapsed = time.time() - start
    
    print(f"   ✓ Error: {result['error']:.2f}% | Compression: {result['compression']:.1f}x | Time: {elapsed:.1f}s")
    
    # Save layer
    layer_file = f"{COMPRESSED_DIR}/layers/{layer_name.replace('.', '_')}.pt"
    torch.save({
        'A': result['A'],
        'B': result['B'],
        'rank': result['rank'],
        'original_shape': result['original_shape']
    }, layer_file)
    
    # Update metadata
    metadata['layers'][layer_name] = {
        'shape': layer_info['shape'],
        'error': result['error'],
        'compression': result['compression'],
        'rank': result['rank'],
        'file': layer_file
    }
    
    # Save metadata after each layer (for resume capability)
    with open(f'{COMPRESSED_DIR}/metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    
    compressed_count += 1
    
    # Progress update every 10 layers
    if (i + 1) % 10 == 0:
        total_elapsed = time.time() - total_start
        print(f"\n📊 Progress: {i+1}/{len(layers_to_compress)} layers | "
              f"Elapsed: {total_elapsed/60:.1f}min\n")

total_time = time.time() - total_start
print("\n" + "="*70)
print("✅ Compression Complete!")
print("="*70)
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Compressed: {compressed_count} layers")
print(f"Skipped: {skipped_count} layers")
print(f"\n📁 Results saved to: {COMPRESSED_DIR}")

## Step 7: Summary & Quality Check

In [ ]:
# Load final metadata
with open(f'{COMPRESSED_DIR}/metadata.json') as f:
    final_metadata = json.load(f)

layers_data = final_metadata['layers']

# Calculate statistics
errors = [l['error'] for l in layers_data.values()]
compressions = [l['compression'] for l in layers_data.values()]

good_layers = [l for l in layers_data.values() if l['error'] <= MAX_ERROR]
bad_layers = [l for l in layers_data.values() if l['error'] > MAX_ERROR]

print("\n" + "="*70)
print("📊 COMPRESSION QUALITY REPORT")
print("="*70)
print(f"\nTotal layers: {len(layers_data)}")
print(f"Good quality (<{MAX_ERROR}% error): {len(good_layers)}")
print(f"Poor quality (>{MAX_ERROR}% error): {len(bad_layers)}")

print(f"\nError statistics:")
print(f"  Mean: {sum(errors)/len(errors):.2f}%")
print(f"  Min: {min(errors):.2f}%")
print(f"  Max: {max(errors):.2f}%")

print(f"\nCompression statistics:")
print(f"  Mean: {sum(compressions)/len(compressions):.1f}x")
print(f"  Min: {min(compressions):.1f}x")
print(f"  Max: {max(compressions):.1f}x")

if bad_layers:
    print(f"\n⚠️ Layers needing re-compression:")
    for name, info in list(layers_data.items())[:5]:
        if info['error'] > MAX_ERROR:
            print(f"  {name}: {info['error']:.2f}% error")

print(f"\n📁 All data saved to Google Drive:")
print(f"   {COMPRESSED_DIR}")
print(f"\n💡 To decompress and benchmark, use the decompress script locally")

## 🔄 Resume Capability

If Colab disconnects:
1. Re-run Steps 1-3 (keep RESUME = True)
2. Run Step 6 again — it will skip already-compressed layers

## 📥 Download Results

You can download the compressed model from Drive:
- Go to `MyDrive/LoRA_Compress/compressed_models/full_model/`
- Download `metadata.json` + `layers/` folder
- Use locally with `scripts/decompress_and_benchmark.py`